# FiNTA — Fine-tune PhoBERT cho Sentiment Analysis

**Chạy trên Google Colab với GPU T4 (miễn phí)**

Pipeline:
1. Kiểm tra GPU
2. Cài đặt thư viện
3. Mount Google Drive (chứa dataset)
4. Load + kiểm tra dataset
5. Tokenize
6. Training
7. Evaluate trên val set
8. Push lên HuggingFace Hub

In [ ]:
# Cell 1 — Kiểm tra GPU
import torch
print(torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Cần GPU! Vào Runtime → Change runtime type → T4"

In [ ]:
# Cell 2 — Install thư viện
!pip install transformers datasets torch accelerate underthesea -q
!pip install huggingface_hub mlflow scikit-learn -q
print("Cài đặt hoàn tất!")

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/finta/dataset_v1"
print(f"Dataset path: {DATASET_PATH}")

In [ ]:
# Cell 4 — Load dataset từ Drive
import json
import pandas as pd

with open(f"{DATASET_PATH}/train.json", encoding="utf-8") as f:
    train_raw = json.load(f)
with open(f"{DATASET_PATH}/val.json", encoding="utf-8") as f:
    val_raw = json.load(f)

train_df = pd.DataFrame(train_raw)
val_df = pd.DataFrame(val_raw)

print(f"Train: {len(train_df)}, Val: {len(val_df)}")
print("\nPhân phối nhãn (Train):")
print(train_df["sentiment_label"].value_counts())

In [ ]:
# Cell 5 — Chuẩn bị Dataset + Oversample class hiếm
import random
import torch
from collections import Counter
from torch.utils.data import Dataset
from transformers import AutoTokenizer

PHOBERT_BASE = "vinai/phobert-base-v2"
MAX_LENGTH = 256

LABEL2ID = {
    "Bullish": 0,
    "Bearish": 1,
    "Uncertainty": 2,
    "Hype/FOMO": 3,
    "Panic": 4,
    "Contrarian": 5,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

tokenizer = AutoTokenizer.from_pretrained(PHOBERT_BASE)


def oversample_rare_classes(records, min_count=60):
    """Lặp lại mẫu của class hiếm cho đến khi đạt min_count."""
    counts = Counter(r["sentiment_label"] for r in records)
    augmented = list(records)
    for label, count in counts.items():
        if count < min_count:
            rare = [r for r in records if r["sentiment_label"] == label]
            needed = min_count - count
            augmented.extend(random.choices(rare, k=needed))
            print(f"  Oversample '{label}': {count} → {count + needed}")
    random.shuffle(augmented)
    return augmented


class FintaSentimentDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records[idx]
        text = f"Tiêu đề: {row.get('title', '')}\n\n{row.get('raw_content', '')}"
        enc = tokenizer(text, max_length=MAX_LENGTH, padding="max_length",
                        truncation=True, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(LABEL2ID[row["sentiment_label"]], dtype=torch.long),
        }


print("Trước oversample:")
print(Counter(r["sentiment_label"] for r in train_raw))

train_aug = oversample_rare_classes(train_raw, min_count=60)

print(f"\nSau oversample: {len(train_aug)} mẫu (trước: {len(train_raw)})")
print(Counter(r["sentiment_label"] for r in train_aug))

train_dataset = FintaSentimentDataset(train_aug)
val_dataset   = FintaSentimentDataset(val_raw)
print(f"\nDataset sẵn sàng: train={len(train_dataset)}, val={len(val_dataset)}")

In [ ]:
# Cell 6 — Training: Focal Loss γ=1, KHÔNG dùng class weights (oversample đã đủ)
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# KHÔNG tính class_weights — oversample đã cân bằng dữ liệu rồi
# Kết hợp oversample + class_weight + Focal Loss γ=2 gây triple-penalization
# làm Bullish bị under-predict nặng (59/98 bài bị đổ sang Uncertainty)

model = AutoModelForSequenceClassification.from_pretrained(
    PHOBERT_BASE,
    num_labels=6,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": round(accuracy_score(labels, preds), 4),
        "macro_f1": round(f1_score(labels, preds, average="macro", zero_division=0), 4),
    }


training_args = TrainingArguments(
    output_dir="./checkpoints/sentiment",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,          # giảm từ 2e-5 → 1e-5 để ổn định hơn
    warmup_ratio=0.15,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    label_smoothing_factor=0.05, # giảm từ 0.1 → 0.05
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    logging_steps=50,
    report_to="none",
    save_total_limit=2,
)


class FocalLossTrainer(Trainer):
    """Focal Loss γ=1 — không class weights, oversample đã xử lý imbalance."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        ce = F.cross_entropy(logits, labels, reduction="none")  # không weight
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** 1 * ce).mean()  # gamma=1
        return (loss, outputs) if return_outputs else loss


trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

print("Bắt đầu training (8 epochs, Focal Loss γ=1, LR=1e-5)...")
trainer.train()

In [ ]:
# Cell 7 — Evaluate trên val set
from sklearn.metrics import classification_report

val_preds = trainer.predict(val_dataset)
y_pred = np.argmax(val_preds.predictions, axis=-1)
y_true = val_preds.label_ids

print("\nCLASSIFICATION REPORT (Validation Set):")
print(classification_report(
    y_true, y_pred,
    target_names=list(ID2LABEL.values()),
    digits=4,
))

# Lưu best model
trainer.save_model("./checkpoints/sentiment/best_model")
tokenizer.save_pretrained("./checkpoints/sentiment/best_model")
print("Best model đã lưu!")

In [ ]:
# Cell 7b — Temperature Scaling (calibration post-training, không cần retrain)
# Tìm temperature T tối ưu trên val set để cải thiện confidence calibration
from scipy.special import softmax as scipy_softmax

val_preds = trainer.predict(val_dataset)
logits_np = val_preds.predictions   # (N, 6)
y_true_val = val_preds.label_ids

# Grid search temperature T ∈ [0.3, 3.0]
best_T, best_macro_f1 = 1.0, 0.0
for T in np.arange(0.3, 3.1, 0.1):
    scaled_probs = scipy_softmax(logits_np / T, axis=-1)
    preds = np.argmax(scaled_probs, axis=-1)
    mf1 = f1_score(y_true_val, preds, average="macro", zero_division=0)
    if mf1 > best_macro_f1:
        best_macro_f1, best_T = mf1, T

print(f"Optimal temperature T = {best_T:.1f}  →  Val Macro F1 = {best_macro_f1:.4f}")

# Áp dụng T và in kết quả val set sau calibration
from sklearn.metrics import classification_report
scaled_probs = scipy_softmax(logits_np / best_T, axis=-1)
y_pred_cal = np.argmax(scaled_probs, axis=-1)

print("\nCLASSIFICATION REPORT (Val set, sau Temperature Scaling):")
print(classification_report(y_true_val, y_pred_cal,
                             target_names=list(ID2LABEL.values()), digits=4))

# Lưu temperature để dùng khi serving
import json
with open("./checkpoints/sentiment/best_model/temperature.json", "w") as f:
    json.dump({"temperature": best_T}, f)
print(f"Đã lưu temperature={best_T} vào best_model/temperature.json")

In [ ]:
# Cell 8 — Push lên HuggingFace Hub
from huggingface_hub import login

HF_TOKEN = "HF_TOKEN_HERE"  # token của bạn
HF_REPO  = "Marky12345/BigData_2026"

login(token=HF_TOKEN)

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print("Model đã được upload thành công!")
print(f"URL: https://huggingface.co/{HF_REPO}")